# AURA Stage 8.3 — Genuine Qwen2.5-3B SQL-Agent Pilot on Kaggle

**Purpose.** Run a bounded free-T4 experiment on **five published AgentSynth `core_v2` SQL tasks** with actual Qwen2.5-3B-Instruct inference, stateful SQLite tool execution, independently captured before/after state, **AURA Stage 7 certificate**, certificate replay, and published-checker reproduction. Export raw evidence for independent audit.

**Research integrity:** scripted Stage 8.2 controls are *not* rerun or counted here. The source tasks are external; model actions must be generated by Qwen on this run. The reproduced published checkers are **NOT the official AgentSynth CLI scorer**. The AURA contracts were pre-authored by Antigravity and may need semantic review. No model is trained. No benchmark labels are revealed to the model. Five runs are an exploratory pilot, **not** a publication-grade accuracy estimate.

**Kaggle settings:** Internet ON, GPU T4 (one T4 sufficient). Use **Run All**. Always download `AURA_STAGE83_QWEN3B_PILOT_RESULTS.zip` from `/kaggle/working/`, even when status is BLOCKED; the diagnostic ZIP is intentional. Do not run the older expansion notebook.

The notebook clones the user's AURA feature branch and pins the external AgentSynth commit for provenance. It does **not** push to GitHub or use paid APIs.

In [ ]:
# Cell 1 — Frozen configuration; does NOT install or download a model
from pathlib import Path
from datetime import datetime, timezone
import sys, os, json, hashlib, time, subprocess, traceback, csv, shutil, zipfile, random

BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
ROOT = BASE / 'aura_stage83_qwen3b_work'
ART = ROOT / 'evidence'
ART.mkdir(parents=True, exist_ok=True)
RESULT_ZIP = BASE / 'AURA_STAGE83_QWEN3B_PILOT_RESULTS.zip'
REPO_URL = 'https://github.com/shanmukhdatta/AURA.git'
AURA_BRANCH = 'experiment/stage-8-3-external-llm-verification'
UPSTREAM_URL = 'https://github.com/agentsynth/agentsynth.git'
UPSTREAM_COMMIT = 'd2564dfb25ed7ba32e367d28117e6afd1ee6c485'
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
MODEL_REVISION = None   # recorded via huggingface_hub.model_info where accessible
SEED = 42
TASKS = ['raise-ticket-priority', 'clearance-discount', 'dedup-keep-latest',
         'cancel-and-void-payment', 'refund-and-restock']
MAX_TURNS = 6
MAX_NEW_TOKENS = 192
# Deliberately one deterministic trial per task. Change to separate sampled seeds ONLY in a new preregistered experiment.
DO_SAMPLE = False

STATE = {'status':'INITIALIZED', 'errors':[], 'episodes':[], 'setup_ok':False, 'gpu_ok':False,
         'model_ok':False, 'started_at_utc':datetime.now(timezone.utc).isoformat(),
         'task_ids':TASKS, 'model_id':MODEL_ID, 'seed':SEED}

def write_json(path, value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,ensure_ascii=False,default=str),encoding='utf-8')

def command(label, args, cwd=None, timeout=300):
    t=time.monotonic()
    proc=subprocess.run(args,cwd=str(cwd) if cwd else None,capture_output=True,text=True,
                        errors='replace',timeout=timeout)
    receipt={'command':args,'cwd':str(cwd) if cwd else None,'exit_code':proc.returncode,
             'elapsed_seconds':round(time.monotonic()-t,4),'stdout_tail':proc.stdout[-5000:],
             'stderr_tail':proc.stderr[-5000:]}
    write_json(ART / 'commands' / f'{label}.json',receipt)
    print(label, 'returncode=',proc.returncode, 'elapsed_s=',receipt['elapsed_seconds'])
    if proc.returncode: raise RuntimeError(f'{label} failed: '+proc.stderr[-2000:])
    return proc.stdout.strip()

write_json(ART/'config.json',{'research_stage':'8.3_gpu_pilot','tasks':TASKS,'seed':SEED,
    'model':MODEL_ID,'max_turns':MAX_TURNS,'max_new_tokens':MAX_NEW_TOKENS,
    'do_sample':DO_SAMPLE,'repo_branch':AURA_BRANCH,'upstream_commit':UPSTREAM_COMMIT,
    'score_label':'reproduced_published_checkers_NOT_official_AgentSynth_CLI'})
print('Evidence directory:',ART)
print('Expected output ZIP:',RESULT_ZIP)


## Gate A — Source and runtime checks **before** model download

This phase fails fast if the original AURA verifier, Stage 8.3 adapter, predeclared contracts, or external task pack are missing. It prevents wasting GPU runtime on incomplete setup.

In [ ]:
# Cell 2 — Setup and source provenance
try:
    import importlib.util
    command('install_python_dependencies', [sys.executable,'-m','pip','install','-q',
       'transformers>=4.47,<5', 'accelerate>=0.32', 'huggingface_hub>=0.26',
       'jsonschema>=4.18','pyyaml>=6.0','pytest>=7'],timeout=360)
    import torch, yaml, transformers, jsonschema
    if not torch.cuda.is_available():
        raise RuntimeError('GPU_NOT_AVAILABLE: enable Kaggle GPU T4; do not use CPU fallback for this 3B pilot')
    STATE['gpu_ok']=True
    REPO=ROOT/'AURA'
    UPSTREAM=ROOT/'agentsynth'
    if not (REPO/'.git').exists():
        command('clone_aura_branch', ['git','clone','--depth','1','--branch',AURA_BRANCH,REPO_URL,str(REPO)],timeout=180)
    repo_sha=command('aura_head_sha',['git','rev-parse','HEAD'],cwd=REPO)
    if not (UPSTREAM/'.git').exists():
        command('clone_agentsynth',['git','clone',UPSTREAM_URL,str(UPSTREAM)],timeout=240)
    command('pin_agentsynth_commit',['git','checkout','--detach',UPSTREAM_COMMIT],cwd=UPSTREAM)
    upstream_sha=command('agentsynth_head_sha',['git','rev-parse','HEAD'],cwd=UPSTREAM)
    assert upstream_sha.strip()==UPSTREAM_COMMIT,'Upstream checkout SHA mismatch'
    # Adapter on the user's branch expects original upstream pack here.
    (REPO/'_deps').mkdir(exist_ok=True)
    link=REPO/'_deps'/'agentsynth'
    if not link.exists():
        os.symlink(UPSTREAM,link,target_is_directory=True)
    assert (REPO/'AURA_Stage7'/'aura_verifier'/'engine.py').is_file()
    assert (REPO/'AURA_Stage7'/'aura_external'/'agentsynth_adapter.py').is_file()
    assert (REPO/'stage83'/'results'/'contracts'/'frozen_contracts_manifest.json').is_file()
    assert (link/'packs'/'core_v2.yaml').is_file()
    manifest=json.loads((REPO/'stage83'/'results'/'contracts'/'frozen_contracts_manifest.json').read_text())
    for task in TASKS:
        assert task in manifest, ('Uncontracted task',task)
        p=REPO/manifest[task]['contract_path']
        assert p.is_file(), ('Missing contract',p)
        recorded=manifest[task].get('contract_sha256')
        if recorded and hashlib.sha256(p.read_bytes()).hexdigest()!=recorded:
            raise RuntimeError('FROZEN_CONTRACT_HASH_MISMATCH: '+task)
    import importlib.metadata as im
    write_json(ART/'environment.json',{'python':sys.version,'torch':torch.__version__,
      'transformers':transformers.__version__,'cuda':torch.version.cuda,
      'gpu':torch.cuda.get_device_name(0),'gpu_count':torch.cuda.device_count(),
      'aura_repository_sha':repo_sha,'agentsynth_upstream_sha':upstream_sha,
      'upstream_pack_sha256':hashlib.sha256((link/'packs'/'core_v2.yaml').read_bytes()).hexdigest(),
      'aura_branch':AURA_BRANCH,'model':MODEL_ID,'hf_revision':MODEL_REVISION})
    sys.path.insert(0,str(REPO/'AURA_Stage7'));sys.path.insert(0,str(REPO))
    from aura_external.agentsynth_adapter import AgentSynthAuraAdapter, compute_frame_diff
    from aura_verifier.engine import verify
    from aura_evidence.capture import SnapshotPair
    scenarios=yaml.safe_load((link/'packs'/'core_v2.yaml').read_text(encoding='utf-8'))
    assert all(t in {s['id'] for s in scenarios} for t in TASKS)
    STATE['setup_ok']=True; STATE['status']='SOURCE_AND_GPU_READY'
    print('SETUP OK; AURA=',repo_sha,'Agentsynth=',upstream_sha,'GPU=',torch.cuda.get_device_name(0))
except Exception as e:
    STATE['status']='BLOCKED_SETUP'
    STATE['errors'].append({'phase':'setup','type':type(e).__name__,'message':str(e),'traceback':traceback.format_exc()})
    print('BLOCKED_SETUP:',str(e))
finally:
    write_json(ART/'run_status.json',STATE)


## Gate B — Independent verifier preflight

Execute a **separate labeled scripted-control probe** first to check that AURA can produce and replay a certificate. This preflight does not count as a real-LLM episode or benchmark success.

In [ ]:
# Cell 3 — Local AURA integration preflight, no model involved
if STATE['setup_ok']:
    try:
        task='raise-ticket-priority'; control_dir=ART/'controls'/'verifier_preflight'
        control=AgentSynthAuraAdapter(task,control_dir)
        ok,obs=control.execute_sql("UPDATE tickets SET priority='high' WHERE id=3")
        assert ok,obs
        result=control.run_aura_verification()
        cert=result['cert']
        probes=json.loads((control_dir/'probes.json').read_text())
        observations=json.loads((control_dir/'observations.json').read_text())
        pair=SnapshotPair(cert['epoch_id'],control.before_snapshot,control.after_snapshot,
                          f'snapshots/{control.run_id}.before.json',f'snapshots/{control.run_id}.after.json',0.0,0)
        cert2=verify(control.contract,probes,observations,run_id=control.run_id,epoch_id=cert['epoch_id'],
                     pair=pair,trust_assumptions=cert['trust_assumptions'])
        assert cert2['verdict']==cert['verdict'] and cert2['obligation_results']==cert['obligation_results']
        assert result['noninterference_verified'],'AURA probe changed DB'
        write_json(control_dir/'preflight_receipt.json',{'control':'SCRIPTED_NOT_LLM','verdict':cert['verdict'],
          'replay_ok':True,'noninterference_ok':True})
        STATE['status']='VERIFIER_READY'; STATE['verifier_ok']=True
        print('AURA preflight passed:', cert['verdict'], 'replay and noninterference passed.')
    except Exception as e:
        STATE['status']='BLOCKED_AURA_PREFLIGHT';STATE['verifier_ok']=False
        STATE['errors'].append({'phase':'aura_preflight','message':str(e),'traceback':traceback.format_exc()})
        print('BLOCKED_AURA_PREFLIGHT:',str(e))
else:
    print('Skipped verifier preflight: setup blocked')
write_json(ART/'run_status.json',STATE)


## Gate C — Load real Qwen2.5-3B-Instruct on T4

No mock-model fallback. We record the actual Hugging Face commit revision if accessible. This is **inference, not training**.

In [ ]:
# Cell 4 — Real GPU model; skip entirely if verifier preflight failed
MODEL=None; TOKENIZER=None
if STATE.get('verifier_ok'):
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM
        from huggingface_hub import model_info
        random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
        t0=time.monotonic()
        TOKENIZER=AutoTokenizer.from_pretrained(MODEL_ID,trust_remote_code=False)
        MODEL=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=torch.float16,
                         trust_remote_code=False,low_cpu_mem_usage=True).to('cuda:0')
        MODEL.eval()
        try: STATE['model_revision']=model_info(MODEL_ID).sha
        except Exception as ex: STATE['model_revision']='UNKNOWN_NETWORK_LOOKUP_FAILED:'+type(ex).__name__
        STATE['model_load_seconds']=round(time.monotonic()-t0,3)
        STATE['model_ok']=True;STATE['status']='MODEL_READY'
        print('QWEN3B REAL MODEL READY',MODEL_ID, 'GPU mem allocated GB',
              round(torch.cuda.memory_allocated()/1024**3,3),'load_s',STATE['model_load_seconds'])
    except Exception as e:
        STATE['status']='BLOCKED_MODEL'; STATE['model_ok']=False
        STATE['errors'].append({'phase':'model','message':str(e),'traceback':traceback.format_exc()})
        print('BLOCKED_MODEL:',str(e))
else: print('Model download skipped: AURA verifier not ready')
write_json(ART/'run_status.json',STATE)


## Gate D — Real generated actions → SQL changes → AURA certificates

Each task is reset to its independent published initial state. The model only sees the task instruction, schema, and its own tool responses. **The full published-checker reproduction is calculated after AURA certification; the inherited AURA adapter itself uses published SQL predicates to construct some readback facts, so these are not fully blinded independent graders.** SQL, tool errors, before/after state, certificate, replay, and reference results are saved. Individual task errors are recorded and the next task is attempted.

In [ ]:
# Cell 5 — Inference and direct AURA verification
import re, sqlite3

def schema_text(db_path):
    with sqlite3.connect(db_path) as c:
        return '\n'.join(x[0] for x in c.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'") if x[0])

def parse_action(text):
    s=text.strip()
    if re.match(r'(?is)^(?:FINAL_ANSWER|Final Answer)\s*:',s):
        return 'final', re.sub(r'(?is)^(?:FINAL_ANSWER|Final Answer)\s*:', '', s).strip()
    match=re.search(r'(?is)```\s*sql\s*\n(.*?)```',s)
    if match: return 'sql',match.group(1).strip()
    match=re.search(r'(?is)^SQL\s*:\s*(.*)',s)
    if match: return 'sql',match.group(1).strip().strip('`')
    return 'final',s

def generate_agent_episode(task_id,scenario,adapter):
    instruction=scenario['task']
    system=("You are a database agent in an isolated SQLite training task. Execute the user's requested database operations. "
       "You have one tool, sql_query(query), which executes one SQLite SQL statement. "
       "First inspect current data as needed; then perform necessary updates and verify. "
       "Output EXACTLY ONE of two formats per turn:\n"
       "For a tool call, output only: ```sql\nONE SQL STATEMENT;\n```\n"
       "For completion, output only: FINAL_ANSWER: <clear concise answer>.\n"
       "Never claim an update succeeded unless the tool confirmed it. "
       "Do not change data beyond what is requested; respect policy refusals.\n\n"
       "Available schema:\n"+schema_text(adapter.db_path))
    history=[{'role':'system','content':system},{'role':'user','content':'Task: '+instruction}]
    actions=[]; final_answer=None; start=time.monotonic()
    for turn in range(MAX_TURNS):
        prompt=TOKENIZER.apply_chat_template(history,tokenize=False,add_generation_prompt=True)
        inputs=TOKENIZER(prompt,return_tensors='pt').to('cuda:0')
        n_input=int(inputs['input_ids'].shape[1]);gt=time.monotonic()
        with torch.inference_mode():
            tokens=MODEL.generate(**inputs,max_new_tokens=MAX_NEW_TOKENS,do_sample=DO_SAMPLE,
                pad_token_id=TOKENIZER.eos_token_id)
        reply=TOKENIZER.decode(tokens[0,n_input:],skip_special_tokens=True).strip()
        action_type,payload=parse_action(reply)
        row={'turn':turn+1,'input_tokens':n_input,'output_tokens':int(tokens.shape[1]-n_input),
             'generation_seconds':round(time.monotonic()-gt,4),'raw_model_response':reply,
             'parsed_type':action_type,'parsed_payload':payload}
        actions.append(row)
        print(f'  turn {turn+1}/{MAX_TURNS}: action={action_type} output_tokens={row["output_tokens"]} gpu_seconds={row["generation_seconds"]}',flush=True)
        history.append({'role':'assistant','content':reply})
        if action_type=='sql':
            success,response=adapter.execute_sql(payload)
            row['tool_ok']=bool(success);row['tool_observation']=response
            history.append({'role':'user','content':
                 'Tool sql_query result (success='+str(success)+'):\n'+json.dumps(response,default=str)+
                 '\nChoose next SQL tool call or FINAL_ANSWER.'})
        else:
            final_answer=payload
            break
    if final_answer is not None: adapter.set_agent_answer(final_answer)
    return {'task_id':task_id,'instruction':instruction,'system_prompt':system,'model_actions':actions,
            'final_answer':final_answer,'total_wall_seconds':round(time.monotonic()-start,4),
            'max_turns':MAX_TURNS,'model_id':MODEL_ID,'model_revision':STATE.get('model_revision'),
            'is_genuine_llm':True,'sampling':DO_SAMPLE,'seed':SEED}

def replay_certificate(adapter,cert,ep_dir):
    pr=json.loads((ep_dir/'probes.json').read_text());ob=json.loads((ep_dir/'observations.json').read_text())
    pair=SnapshotPair(cert['epoch_id'],adapter.before_snapshot,adapter.after_snapshot,
       f'snapshots/{adapter.run_id}.before.json',f'snapshots/{adapter.run_id}.after.json',0.0,0)
    r=verify(adapter.contract,pr,ob,run_id=adapter.run_id,epoch_id=cert['epoch_id'],
             pair=pair,trust_assumptions=cert['trust_assumptions'])
    return {'verified':r['verdict']==cert['verdict'] and
            r['obligation_results']==cert['obligation_results'] and r['reason_codes']==cert['reason_codes'],
            'original_verdict':cert['verdict'],'replay_verdict':r['verdict']}

def reproduced_upstream_checkers(scenario,adapter):
    # NOT the official AgentSynth CLI evaluator! Complete *published* task checkers are reproduced.
    details=[]
    for i,c in enumerate(scenario['checkers']):
        if c['kind']=='sql':
            try:
                with sqlite3.connect(f'file:{adapter.db_path}?mode=ro',uri=True) as db:
                    observed=[list(x) for x in db.execute(c['query']).fetchall()]
                matches=observed==c['equals']
            except Exception as ex:
                observed={'error':str(ex)};matches=False
        elif c['kind']=='called_tool':
            observed=[x.get('tool') for x in adapter.trace]
            matches=any(x.get('tool')==c['name'] for x in adapter.trace)
        elif c['kind']=='answer':
            observed=adapter.agent_answer or ''
            matches=any(str(w).casefold() in observed.casefold() for w in c['any_of'])
        else:
            observed='unsupported_checker_kind';matches=False
        details.append({'index':i,'kind':c['kind'],'matched':matches,'observed':observed,'reference_spec':c})
    return {'method':'REPRODUCED_PUBLISHED_CHECKERS_NOT_OFFICIAL_AGENTSYNTH_RUNNER',
       'all_matched':all(x['matched'] for x in details),'passed_count':sum(x['matched'] for x in details),
       'total_count':len(details),'checks':details}

if STATE['model_ok']:
    scenario_map={x['id']:x for x in scenarios}
    for task_id in TASKS:
        run_id=f'qwen3b_real_{task_id}_seed{SEED}'
        ep_dir=ART/'episodes'/task_id/run_id
        try:
            if ep_dir.exists(): shutil.rmtree(ep_dir)
            ep_dir.mkdir(parents=True,exist_ok=True)
            print('RUNNING REAL QWEN TASK:',task_id,flush=True)
            adapter=AgentSynthAuraAdapter(task_id,ep_dir)
            adapter.run_id=run_id
            trajectory=generate_agent_episode(task_id,scenario_map[task_id],adapter)
            write_json(ep_dir/'model_trajectory.json',trajectory)
            write_json(ep_dir/'agent_tool_trace.json',adapter.trace)
            # AURA certificate produced before reading the published checker labels.
            aura_result=adapter.run_aura_verification()
            cert=aura_result['cert']
            replay=replay_certificate(adapter,cert,ep_dir)
            write_json(ep_dir/'certificate_replay.json',replay)
            assert replay['verified'],'CERTIFICATE_REPLAY_MISMATCH'
            assert aura_result['noninterference_verified'],'INSPECTION_MUTATED_DATABASE'
            # Only now compute independent full published checker reproduction.
            external_ref=reproduced_upstream_checkers(scenario_map[task_id],adapter)
            write_json(ep_dir/'published_checker_reproduction.json',external_ref)
            receipt={'task_id':task_id,'run_id':run_id,'is_real_llm':True,
                     'sql_tool_calls':sum(x.get('tool')=='sql_query' for x in adapter.trace),
                     'aura_verdict':cert['verdict'],'certificate_replay':True,
                     'snapshot_noninterference':True,
                     'reproduced_full_published_checker_pass':external_ref['all_matched'],
                     'NOTE':'Reproduced checker comparison NOT official AgentSynth runner result',
                     'trajectory_wall_seconds':trajectory['total_wall_seconds'],
                     'state':'EXECUTED_AND_AURA_EVALUATED'}
            write_json(ep_dir/'run_receipt.json',receipt)
            STATE['episodes'].append(receipt)
            print(task_id,'AURA',cert['verdict'],'FULL PUBLISHED CHECKERS REPRODUCED',external_ref['all_matched'])
        except Exception as e:
            err={'task_id':task_id,'phase':'episode','error_type':type(e).__name__,
                 'message':str(e),'traceback':traceback.format_exc()}
            STATE['errors'].append(err)
            write_json(ep_dir/'ERROR.json',err)
            STATE['episodes'].append({'task_id':task_id,'run_id':run_id,
                        'state':'ERROR_NOT_VALID_EVALUATION','error':str(e)})
            print('FAILED task:',task_id,str(e))
        finally:
            write_json(ART/'run_status.json',STATE)
    STATE['status']='COMPLETED_PILOT' if all(e.get('state')=='EXECUTED_AND_AURA_EVALUATED'
                                         for e in STATE['episodes']) else 'PARTIAL_PILOT'
else:
    print('LLM task execution skipped (setup/verifier/model blocked); final cell still exports ZIP')
write_json(ART/'run_status.json',STATE)


## Gate E — Auditable ZIP export (always run)

The export includes only the new pilot's evidence, not model weights, environment snapshots unrelated to tasks, historical scripts, secrets, or the cloned Git repositories.

In [ ]:
# Cell 6 — Results table, integrity manifest and final ZIP
from collections import Counter
try:
    STATE['finished_at_utc']=datetime.now(timezone.utc).isoformat()
    write_json(ART/'run_status.json',STATE)
    rows=[]
    for e in STATE['episodes']:
        rows.append({k:e.get(k,'') for k in ['task_id','run_id','state','sql_tool_calls','aura_verdict',
            'reproduced_full_published_checker_pass','certificate_replay','snapshot_noninterference',
            'trajectory_wall_seconds','error']})
    cols=['task_id','run_id','state','sql_tool_calls','aura_verdict',
          'reproduced_full_published_checker_pass','certificate_replay','snapshot_noninterference',
          'trajectory_wall_seconds','error']
    with (ART/'pilot_results.csv').open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=cols);w.writeheader();w.writerows(rows)
    counts=Counter(e.get('aura_verdict') for e in STATE['episodes'] if e.get('state')=='EXECUTED_AND_AURA_EVALUATED')
    report=f"""# AURA Stage 8.3 Qwen2.5-3B pilot export

- Run status: {STATE['status']}
- Model: {MODEL_ID}, revision: {STATE.get('model_revision','NOT_LOADED')}
- Tasks scheduled: {len(TASKS)}; real-model AURA-evaluated: {sum(x.get('state')=='EXECUTED_AND_AURA_EVALUATED' for x in STATE['episodes'])}
- Verdict counts among completed: {dict(counts)}
- Setup/model/task errors: {len(STATE['errors'])}
- Dataset source commit: {UPSTREAM_COMMIT}
- Reference label: **REPRODUCED_PUBLISHED_CHECKERS_NOT_OFFICIAL_AGENTSYNTH_RUNNER**.
- All state-only contracts must be reviewed for completeness. These episodes are NOT a random sample or proof of general accuracy.
- AURA certificates and replay receipts are separate from published-checker reproduction.
- Model-generated SQL actions are kept separate from prior scripted controls.
- No training or Git push performed.
"""
    (ART/'README_RESULTS.md').write_text(report,encoding='utf-8')
    # Include raw SQLite as well as evidence snapshots; no symlinks or external files.
    entries=[]
    for p in sorted(ART.rglob('*')):
        if p.is_file() and p.name!='FILE_MANIFEST_SHA256.json':
            entries.append({'path':p.relative_to(ART).as_posix(),'bytes':p.stat().st_size,
                            'sha256':hashlib.sha256(p.read_bytes()).hexdigest()})
    write_json(ART/'FILE_MANIFEST_SHA256.json',{'hash_method':'sha256','files':entries})
    if RESULT_ZIP.exists(): RESULT_ZIP.unlink()
    with zipfile.ZipFile(RESULT_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=7) as z:
        for p in sorted(ART.rglob('*')):
            if p.is_file(): z.write(p,'results/'+p.relative_to(ART).as_posix())
    with zipfile.ZipFile(RESULT_ZIP) as z:
        assert z.testzip() is None
        for r in entries:
            assert hashlib.sha256(z.read('results/'+r['path'])).hexdigest()==r['sha256']
    print('FINAL EXPORT:',RESULT_ZIP,'bytes:',RESULT_ZIP.stat().st_size)
    print('STATUS:',STATE['status'],'Completed model episodes:',len([e for e in STATE['episodes'] if e.get('state')=='EXECUTED_AND_AURA_EVALUATED']))
    print('Verdicts:',dict(counts),'errors:',len(STATE['errors']),'manifest_files:',len(entries))
except Exception as e:
    print('EXPORT ERROR:',type(e).__name__,str(e))
    traceback.print_exc()
    raise
